# Agent 核心算法：ReAct / CoT / 自我反思 / Tree-of-Thought

本 Notebook **自包含、可独立运行**，仅依赖 Python 标准库（`re` / `json` / `itertools` / `collections`），**不调用任何真实 LLM 或外部 API**。

为了让 Agent 流程能离线跑通，我们做两处「hack」：

1. **工具 hack**：用本地字典 / 安全算术求值来模拟真实的搜索引擎与计算器 API。
2. **LLM hack**：用手写的规则策略（mock LLM）来模拟「大模型生成下一步」，把 Agent 的**控制流**清晰地暴露出来。

涵盖内容：

1. 工具定义与「工具 hack」
2. 工具调用解析（Tool-Call Parsing）
3. Chain-of-Thought（思维链）
4. ReAct 框架（手写 demo）
5. 自我反思（Reflexion / Self-Reflection）
6. Tree-of-Thought（思维树搜索）

## 参考
- Yao et al., 2022, *ReAct: Synergizing Reasoning and Acting in Language Models*
- Wei et al., 2022, *Chain-of-Thought Prompting Elicits Reasoning in LLMs*
- Shinn et al., 2023, *Reflexion: Language Agents with Verbal Reinforcement Learning*
- Yao et al., 2023, *Tree of Thoughts: Deliberate Problem Solving with LLMs*

## 1. 工具定义与「工具 hack」

真实 Agent 会调用搜索引擎、数据库、代码解释器等。这里我们用**本地实现**来替代，保证离线可运行：

- `search`：在一个本地事实字典里做关键词匹配（模拟检索）。
- `calculator`：用正则白名单 + 受限 `eval` 做**安全**的算术求值。

In [ ]:
import re
import json
import itertools
from collections import deque

# ---- 工具 hack：用本地字典 / 安全计算模拟真实 API ----
_FACTS = {
    "capital of france": "Paris",
    "capital of japan": "Tokyo",
    "population of paris": "2161000",
    "population of tokyo": "13960000",
    "height of mount everest": "8849",
}


def search(query: str) -> str:
    # 模拟搜索引擎/知识库：命中关键词就返回事实，否则返回未找到
    q = query.strip().lower()
    for key, val in _FACTS.items():
        if key in q:
            return val
    return "No result found."


def calculator(expr: str) -> str:
    # 安全算术求值：只允许数字与 + - * / ( ) . 和空格，杜绝任意代码执行
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr):
        return "Error: invalid expression"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


# 工具注册表：名字 -> 可调用对象
TOOLS = {"search": search, "calculator": calculator}

print("search('capital of France') =>", search("capital of France"))
print("calculator('2161000 * 2')   =>", calculator("2161000 * 2"))
print("calculator('__import__')    =>", calculator("__import__('os')"))

## 2. 工具调用解析（Tool-Call Parsing）

LLM 的输出是**纯文本**，我们需要从中解析出「调用哪个工具、传什么参数」。这里支持两种常见格式：

- **方括号式**（ReAct 经典写法）：`Action: search[capital of France]`
- **JSON 式**（function-calling 写法）：`Action: {"tool": "search", "input": "capital of France"}`

In [ ]:
# 方括号式：Action: name[input]
ACTION_BRACKET_RE = re.compile(r"Action:\s*([A-Za-z_]\w*)\[(.*?)\]", re.DOTALL)
# JSON 式：Action: {...}
ACTION_JSON_RE = re.compile(r"Action:\s*(\{.*?\})", re.DOTALL)
# 思维行：Thought: ...
THOUGHT_RE = re.compile(r"Thought:\s*(.*)")


def parse_action(text: str):
    # 返回 (tool_name, tool_input)；解析失败返回 None。优先取最后一个 Action
    matches = list(ACTION_BRACKET_RE.finditer(text))
    if matches:
        m = matches[-1]
        return m.group(1), m.group(2).strip()
    mj = ACTION_JSON_RE.search(text)
    if mj:
        try:
            obj = json.loads(mj.group(1))
            return obj.get("tool"), str(obj.get("input", "")).strip()
        except json.JSONDecodeError:
            return None
    return None


def parse_thought(text: str):
    m = THOUGHT_RE.search(text)
    return m.group(1).strip() if m else ""


# 演示两种格式都能解析
print(parse_action("Thought: 先查首都\nAction: search[capital of France]"))
print(parse_action('Action: {"tool": "calculator", "input": "1+2*3"}'))
print("thought:", parse_thought("Thought: 我需要先检索\nAction: search[x]"))

## 3. Chain-of-Thought（思维链）

CoT 的核心：让模型**显式写出中间推理步骤**，而不是直接给答案。对需要多步推理的问题，这能显著降低出错率。

下面用一个应用题对比「直接作答」与「逐步推理」。

In [ ]:
PROBLEM = "小明原有 3 个苹果，又买了 2 袋，每袋 4 个，一共有多少个苹果？"


def solve_direct(problem):
    # 不做推理，凭直觉“蒙”一个（常见错误：把 3+2 当成一步，再乘 4 => 20）
    return 20


def solve_cot(problem):
    # Chain-of-Thought：显式写出每一步
    steps = [
        "原有苹果：3 个",
        "新买苹果：2 袋 × 每袋 4 个 = 8 个",
        "合计：3 + 8 = 11 个",
    ]
    for s in steps:
        print("   -", s)
    return 11


print("问题:", PROBLEM)
print("[直接作答] =>", solve_direct(PROBLEM), "(错误)")
print("[思维链 CoT]")
ans = solve_cot(PROBLEM)
print("[思维链 CoT] =>", ans, "(正确)")

## 4. ReAct 框架（手写 demo）

**ReAct = Reasoning + Acting**：模型交替产生 `Thought`（推理）和 `Action`（调用工具），把工具返回的 `Observation` 拼回上下文，循环直至 `finish`。

```
Thought -> Action -> Observation -> Thought -> Action -> ... -> finish
```

由于不接真实 LLM，我们**手写一个规则策略 `mock_react_llm`** 来扮演大模型：它根据已有的 Observation 决定下一步动作——这正是 ReAct 控制流的核心。

In [ ]:
def mock_react_llm(question, scratchpad):
    # “手写”的策略：根据已获得的 Observation 数量决定下一步
    # 真实场景中，这里是把 (question + scratchpad) 拼成 prompt 喂给 LLM 生成
    obs = [content for role, content in scratchpad if role == "Observation"]
    n = len(obs)
    if n == 0:
        return "Thought: 我需要先查出法国的首都。\nAction: search[capital of France]"
    if n == 1:
        city = obs[0]
        return f"Thought: 首都是 {city}，接着查它的人口。\nAction: search[population of {city}]"
    if n == 2:
        pop = obs[1]
        return f"Thought: 人口是 {pop}，题目要求乘以 2。\nAction: calculator[{pop} * 2]"
    return f"Thought: 计算结果是 {obs[2]}，可以给出最终答案了。\nAction: finish[{obs[2]}]"


def react_agent(question, policy, tools, max_steps=6, verbose=True):
    # 通用 ReAct 循环：policy 生成 Thought+Action，工具执行得到 Observation
    scratchpad = []
    if verbose:
        print("Question:", question)
    for step in range(1, max_steps + 1):
        llm_out = policy(question, scratchpad)
        thought = parse_thought(llm_out)
        parsed = parse_action(llm_out)
        if verbose:
            print(f"\n[Step {step}]")
            print("  Thought:", thought)
        if parsed is None:
            if verbose:
                print("  (无法解析出 Action，终止)")
            return None
        name, arg = parsed
        if verbose:
            print(f"  Action : {name}[{arg}]")
        if name == "finish":
            if verbose:
                print("  => Final Answer:", arg)
            return arg
        obs = tools[name](arg) if name in tools else f"Unknown tool: {name}"
        if verbose:
            print("  Observation:", obs)
        scratchpad.append(("Action", f"{name}[{arg}]"))
        scratchpad.append(("Observation", obs))
    return None


final = react_agent("法国首都的人口是多少？再乘以 2 等于多少？",
                    mock_react_llm, TOOLS)
print("\n返回值:", final)

## 5. 自我反思（Reflexion / Self-Reflection）

Reflexion 的思路：Agent 先尝试 → 外部/自我**评估**结果 → 把失败原因写成**反思文本**存入记忆 → 带着反思**重试**。
反思相当于一种「语言形式的强化信号」，无需更新参数即可改进后续行为。

下面让 Agent 解一道运算优先级容易出错的题：第一次没有反思 → 算错；带着反思 → 改对。

In [ ]:
TARGET = 11  # 3 + 2*4 的正确答案


def attempt(problem, reflections):
    # mock LLM：没有反思时犯“忽略运算优先级”的错误；有反思后纠正
    if not reflections:
        return (3 + 2) * 4          # 20，错误
    return 3 + 2 * 4                # 11，正确


def evaluate(answer, target):
    return answer == target


def reflexion_loop(problem, target, max_trials=3):
    reflections = []
    for trial in range(1, max_trials + 1):
        ans = attempt(problem, reflections)
        ok = evaluate(ans, target)
        print(f"[Trial {trial}] answer={ans}  correct={ok}")
        if ok:
            return ans
        # 生成反思（真实场景由 LLM 根据错误产生）
        note = "上次错误：没有遵循运算优先级，应先做乘除再做加减。"
        reflections.append(note)
        print("   reflection:", note)
    return ans


reflexion_loop("计算 3 + 2 * 4", TARGET)

## 6. Tree-of-Thought（思维树搜索）

ToT 把推理建模成在**思维树**上的搜索：每个节点是一个中间状态，模型生成多个候选「思维」作为分支，再用一个**评估器**对分支打分/剪枝，并用 BFS/DFS/Beam 等策略搜索。

我们用经典的 **24 点游戏**演示：给定一组数字，每次任选两个数做一次四则运算合成新数，目标是最终只剩一个数且等于 24。
- **思维节点**：当前剩余数字的集合
- **思维分支**：任选两数 + 一种运算
- **搜索策略**：BFS（并记录路径，便于展示推理链）

In [ ]:
def tot_solve_24(numbers, target=24, tol=1e-6):
    # 每个状态是“剩余数字”的有序元组；扩展=任选两数做一次运算（即生成思维分支）
    start = tuple(sorted(float(n) for n in numbers))
    queue = deque([(start, [])])
    seen = set()
    while queue:
        state, path = queue.popleft()
        if len(state) == 1:
            if abs(state[0] - target) < tol:
                return path                       # 找到一条到达目标的思维链
            continue
        if state in seen:
            continue
        seen.add(state)
        nums = list(state)
        for i, j in itertools.permutations(range(len(nums)), 2):
            a, b = nums[i], nums[j]
            rest = [nums[k] for k in range(len(nums)) if k != i and k != j]
            # 候选运算 = 思维分支（评估器在此可做剪枝，这里保留全部分支）
            branches = [(a + b, f"{a:g}+{b:g}"),
                        (a - b, f"{a:g}-{b:g}"),
                        (a * b, f"{a:g}*{b:g}")]
            if abs(b) > tol:
                branches.append((a / b, f"{a:g}/{b:g}"))
            for val, desc in branches:
                new_state = tuple(sorted(rest + [val]))
                queue.append((new_state, path + [f"{desc}={val:g}"]))
    return None


for nums in ([2, 3, 4], [4, 9, 10, 1], [8, 8, 3, 3]):
    sol = tot_solve_24(nums)
    print(f"numbers={nums} -> ", " ; ".join(sol) if sol else "无解")

## 7. 小结

| 方法 | 一句话核心 | 适用场景 |
| --- | --- | --- |
| CoT | 显式写出中间推理步骤 | 多步推理 / 算术 / 常识 |
| ReAct | 推理与工具调用交替（Thought→Action→Observation） | 需要外部知识/工具的任务 |
| Reflexion | 失败后写反思并重试 | 可验证结果、允许多次尝试 |
| Tree-of-Thought | 在思维树上生成分支 + 评估 + 搜索 | 需要探索/回溯的难题（如 24 点、规划） |

本 Notebook 用「工具 hack + mock LLM」把这些方法的**控制流**完整跑通；接入真实 LLM 时，只需把 `mock_react_llm` / `attempt` 等替换为对模型的调用，把 `TOOLS` 换成真实工具即可。